# Word sentiment EEG — short runner

Use a **CPU** Colab runtime and choose **Runtime → Run all**. Allow Drive mounting and access to the existing `GITHUB_TOKEN` secret. That token must be allowed to read `parmisbathayan/word-sentiment-eeg`.

This run reuses ZuCo 1.0 **SR + NR + TSR** word caches, loads the already-prepared agent word labels, validates exact EEG alignment, exports class EEG, and runs controlled EEG-only screens. Raw fixation segments are exported from existing MATLAB files where available. No missing EEG is downloaded.

All work runs in Colab. Results stay in `MyDrive/Thesis/Results/word_sentiment_eeg/v2_agent_annotated`; reusable raw outputs stay under `CachedArtifacts/word_sentiment_eeg`. The annotation table already covers all 1,037 distinct sentences and 20,458 source word occurrences. Labels are single-agent judgments, not human consensus. The primary screen uses agreement between lexical and contextual labels; uncertainty stays explicit. Scientific results are pending until this run finishes.


In [ ]:
# Run all. This cell runs only in Colab; nothing is installed on your Mac.
from google.colab import drive, userdata
from pathlib import Path
import base64, importlib.metadata, json, os, subprocess, sys
from IPython.display import Markdown, display

drive.mount('/content/drive')
THESIS = Path('/content/drive/MyDrive/Thesis')
REPO = Path('/content/word-sentiment-eeg')
REPO_URL = 'https://github.com/parmisbathayan/word-sentiment-eeg.git'
RUN_TAG = 'v2_agent_annotated'
EXPORT_AVAILABLE_RAW = True  # reads existing MATLAB files; never downloads missing EEG

try:
    token = userdata.get('GITHUB_TOKEN')
except Exception as error:
    raise RuntimeError('Allow the existing GITHUB_TOKEN Colab secret to read parmisbathayan/word-sentiment-eeg, then rerun this cell.') from None
if not token:
    raise RuntimeError('The GITHUB_TOKEN Colab secret is empty.')
header = 'AUTHORIZATION: basic ' + base64.b64encode(('x-access-token:' + token).encode()).decode()
git_env = dict(os.environ, GIT_CONFIG_COUNT='1', GIT_CONFIG_KEY_0='http.https://github.com/.extraheader', GIT_CONFIG_VALUE_0=header, GIT_TERMINAL_PROMPT='0')
def git(*args):
    result = subprocess.run(['git', *map(str, args)], env=git_env, capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError('GitHub access/update failed. Check token access to this repository and any local Colab code changes. No Drive results were removed.')
if not REPO.exists():
    git('clone', '--depth', '1', '--branch', 'main', REPO_URL, REPO)
else:
    if subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip():
        raise RuntimeError('The Colab code checkout has edits. Save them before updating the checkout.')
    git('-C', REPO, 'pull', '--ff-only', 'origin', 'main')
del token, header, git_env
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements-colab.txt')], check=True)
print('Code version:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', '--short', 'HEAD'], text=True).strip())
print('CPU is sufficient. Existing EEG caches stay in place on Drive.')


In [ ]:
# Load prepared annotations, validate alignment, export class EEG, and evaluate. Progress and outputs save to Drive.
command = [sys.executable, '-u', str(REPO / 'main.py'), '--thesis-root', str(THESIS), '--run-tag', RUN_TAG]
if EXPORT_AVAILABLE_RAW:
    command += ['--raw-dir', str(THESIS / 'Data/zuco_og_raw')]
subprocess.run(command, check=True)


In [ ]:
RUN = THESIS / 'Results/word_sentiment_eeg' / RUN_TAG
if (RUN / 'REPORT.md').exists():
    display(Markdown((RUN / 'REPORT.md').read_text()))
else:
    print('The run is incomplete. See', RUN / 'run.log')
if (RUN / 'plots/class_coverage.png').exists():
    from IPython.display import Image
    display(Image(filename=str(RUN / 'plots/class_coverage.png')))
print('Results:', RUN)
for folder in [RUN, THESIS / 'CachedArtifacts/word_sentiment_eeg']:
    total = sum(p.stat().st_size for p in folder.rglob('*') if p.is_file()) if folder.exists() else 0
    print(f'{folder}: {total / 1024**3:.3f} GiB')
print('When finished, tell Codex: the word-sentiment run finished. Codex can read REPORT.md and the small result files through Drive.')
